# Lab 02: the memory wall

Watch the GPU read its weights once per token, and see why the utilisation figure says nothing. Uses the engine in-process rather than as a server.

In [ ]:
# Cell 1: the GPU this notebook got. On Kaggle: Settings → Accelerator → GPU T4 x2, then Run all.
import subprocess, sys
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
if out.returncode != 0 or not out.stdout.strip():
    sys.exit("No GPU. Open Settings → Accelerator and choose GPU T4 x2, then run this cell again.")
print(out.stdout.strip())
if "T4" in out.stdout:
    print("Path: Kaggle T4. Labs 08 and 10 are read-only on this GPU (no FP8 arithmetic).")

In [ ]:
# Cell 2: install the pinned engine. About nine minutes on Kaggle. Kaggle's preinstalled torchaudio is built
# for an older CUDA and breaks the import, so it is removed.
%pip install -q "vllm==0.32.0" 2>&1 | tail -2
%pip uninstall -y -q torchaudio
import vllm, torch
print("vllm", vllm.__version__, "· torch", torch.__version__, "· compute capability", torch.cuda.get_device_capability())

In [ ]:
# Cell 3: load the model once and measure the decode step at batch 1 and batch 32. On the T4 the weights are fp16: the same 8.04 GB.
import time, subprocess, threading
from vllm import LLM, SamplingParams
MODEL = "Qwen/Qwen3-4B-Instruct-2507"; WEIGHT_BYTES = 8.04e9; SPEC_GBPS = 320   # the T4's memory bandwidth
llm = LLM(MODEL, dtype="half", max_model_len=4096, gpu_memory_utilization=0.9, enable_prefix_caching=False, seed=0)
prompts = [f"Write a short paragraph about the number {i}." for i in range(32)]
llm.generate(prompts[:1], SamplingParams(max_tokens=8, temperature=0, ignore_eos=True))
for bs in (1, 32):
    ps = prompts[:bs]
    t1 = time.time(); llm.generate(ps, SamplingParams(max_tokens=1, temperature=0, ignore_eos=True)); t_one = time.time() - t1
    t1 = time.time(); llm.generate(ps, SamplingParams(max_tokens=257, temperature=0, ignore_eos=True)); t_many = time.time() - t1
    step = (t_many - t_one) / 256
    print(f"batch {bs}: decode step {step * 1000:.1f} ms, {bs / step:.0f} tokens/s; implied read rate {WEIGHT_BYTES / step / 1e9:.0f} GB/s of {SPEC_GBPS}; arithmetic {2 * 4.02e9 * bs / step / 1e12:.1f} TFLOP/s of 65")

In [ ]:
# Cell 4: what nvidia-smi says during a long batch-1 decode.
def dmon(seconds):
    f = open("/kaggle/working/dmon.txt", "w"); p = subprocess.Popen(["nvidia-smi", "dmon", "-s", "um", "-d", "1"], stdout=f, stderr=subprocess.STDOUT)
    time.sleep(seconds); p.terminate(); p.wait(); f.close()
    rows = []
    for line in open("/kaggle/working/dmon.txt"):
        parts = line.split()
        if parts and parts[0] != "#":
            try: rows.append((float(parts[1]), float(parts[2])))
            except (ValueError, IndexError): pass
    return rows[1:]
rows = []; t = threading.Thread(target=lambda: rows.extend(dmon(8))); t.start(); time.sleep(1.0)
llm.generate(prompts[:1], SamplingParams(max_tokens=200, temperature=0, ignore_eos=True)); t.join()
print("second  sm%  mem%")
for i, (sm, mem) in enumerate(rows[:8]): print(f"{i + 1:>6}  {sm:4.0f}  {mem:4.0f}")